<a href="https://colab.research.google.com/github/saim2003tech/ARTIFICIAL-INTELLIGENCE/blob/main/AI_LAB3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from collections import deque
import heapq
import random
import matplotlib.pyplot as plt

# Maze: S = start, G = goal, # = wall
MAZE = [
    "S . . . . # #",
    "# # . # # . .",
    ". . . # . . #",
    ". # . . . . #",
    ". . . . # . G"
]
grid = [row.split() for row in MAZE]
ROWS, COLS = len(grid), len(grid[0])

def find(ch, board=grid):
    for r, row in enumerate(board):
        for c, value in enumerate(row):
            if value == ch:
                return (r, c)
    return None

def neighbours(cell, order=None, board=grid):
    if order is None:
        order = [(-1, 0), (0, 1), (1, 0), (0, -1)]  # Up, Right, Down, Left
    r, c = cell
    for dr, dc in order:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(board) and 0 <= nc < len(board[0]) and board[nr][nc] != '#':
            yield (nr, nc)

def build_path(parent, node):
    path = []
    while node is not None:
        path.append(node)
        node = parent[node]
    return path[::-1]

def bfs(start, goal, board=grid, order=None):
    frontier = deque([start])
    parent = {start: None}
    expanded = 0
    max_frontier = 1
    while frontier:
        node = frontier.popleft()
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node, order, board):
            if nb not in parent:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

def dfs(start, goal, board=grid, order=None):
    frontier = [start]
    parent = {start: None}
    visited = set()
    expanded = 0
    max_frontier = 1
    while frontier:
        node = frontier.pop()
        if node in visited:
            continue
        visited.add(node)
        expanded += 1
        if node == goal:
            return build_path(parent, goal), expanded, max_frontier
        for nb in neighbours(node, order, board):
            if nb not in visited and nb not in parent:
                parent[nb] = node
                frontier.append(nb)
        max_frontier = max(max_frontier, len(frontier))
    return None, expanded, max_frontier

S, G = find('S'), find('G')
results = {}
for name, algorithm in [('BFS', bfs), ('DFS', dfs)]:
    path, expanded, max_frontier = algorithm(S, G)
    results[name] = (path, expanded, max_frontier)
    print(f"{name}: path length={len(path)-1 if path else 'No path'}, nodes expanded={expanded}, max frontier={max_frontier}")

BFS: path length=10, nodes expanded=23, max frontier=5
DFS: path length=16, nodes expanded=17, max frontier=5


In [5]:
print(f"{'Algorithm':<10}{'Path length':>14}{'Nodes expanded':>18}{'Max frontier':>16}")
for name, (path, expanded, max_frontier) in results.items():
    print(f"{name:<10}{len(path)-1:>14}{expanded:>18}{max_frontier:>16}")

Algorithm    Path length    Nodes expanded    Max frontier
BFS                   10                23               5
DFS                   16                17               5


In [6]:
def print_maze_with_path(path, board=grid):
    shown = [row[:] for row in board]
    if path:
        for r, c in path:
            if shown[r][c] not in ('S', 'G'):
                shown[r][c] = '*'
    for row in shown:
        print(' '.join(row))

for name, (path, _, _) in results.items():
    print(f"\\n{name} path:")
    print_maze_with_path(path)

\nBFS path:
S * * . . # #
# # * # # . .
. . * # . . #
. # * * * * #
. . . . # * G
\nDFS path:
S * * . . # #
# # * # # . .
* * * # . . #
* # . * * * #
* * * * # * G


In [7]:
left_down_right_up = [(0, -1), (1, 0), (0, 1), (-1, 0)]
for name, algorithm in [('BFS', bfs), ('DFS', dfs)]:
    path, expanded, max_frontier = algorithm(S, G, order=left_down_right_up)
    print(f"{name} with Left, Down, Right, Up: length={len(path)-1 if path else None}, expanded={expanded}, max frontier={max_frontier}")
    print_maze_with_path(path)

BFS with Left, Down, Right, Up: length=10, expanded=22, max frontier=4
S * * . . # #
# # * # # . .
. . * # . . #
. # * * * * #
. . . . # * G
DFS with Left, Down, Right, Up: length=10, expanded=17, max frontier=5
S * * . . # #
# # * # # . .
. . * # . . #
. # * * * * #
. . . . # * G


In [8]:
graph = {
    'Islamabad': {'Rawalpindi': 15, 'Murree': 60, 'Attock': 80, 'Chakwal': 120},
    'Rawalpindi': {'Islamabad': 15, 'Chakwal': 90, 'Jhelum': 110},
    'Murree': {'Islamabad': 60, 'Abbottabad': 65},
    'Attock': {'Islamabad': 80, 'Peshawar': 100},
    'Chakwal': {'Rawalpindi': 90, 'Lahore': 230, 'Islamabad': 120, 'Gujranwala': 150},
    'Jhelum': {'Rawalpindi': 110, 'Gujranwala': 110},
    'Gujranwala': {'Jhelum': 110, 'Lahore': 70, 'Chakwal': 150},
    'Abbottabad': {'Murree': 65},
    'Peshawar': {'Attock': 100},
    'Lahore': {'Chakwal': 230, 'Gujranwala': 70}
}

def ucs(graph, start, goal):
    frontier = [(0, start, [start])]
    best_cost = {start: 0}
    expanded = 0
    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if cost != best_cost.get(node):
            continue
        expanded += 1
        if node == goal:
            return path, cost, expanded
        for nb, weight in graph[node].items():
            new_cost = cost + weight
            if new_cost < best_cost.get(nb, float('inf')):
                best_cost[nb] = new_cost
                heapq.heappush(frontier, (new_cost, nb, path + [nb]))
    return None, float('inf'), expanded

path, cost, expanded = ucs(graph, 'Islamabad', 'Lahore')
print('UCS path:', ' -> '.join(path))
print('UCS cost:', cost, 'km')
print('Nodes expanded:', expanded)

UCS path: Islamabad -> Rawalpindi -> Jhelum -> Gujranwala -> Lahore
UCS cost: 305 km
Nodes expanded: 10


In [9]:
maze10 = [
    list('S..m......'),
    list('.##m.##...'),
    list('...m...m..'),
    list('##...##m..'),
    list('...m......'),
    list('.m.##.m...'),
    list('...m...##.'),
    list('##...m....'),
    list('...##..m..'),
    list('......m..G')
]
# Ensure dimensions are 10x10
assert len(maze10) == 10 and all(len(row) == 10 for row in maze10)
start10 = find('S', maze10); goal10 = find('G', maze10)

def maze10_neighbours(cell):
    r, c = cell
    for dr, dc in [(-1,0),(0,1),(1,0),(0,-1)]:
        nr, nc = r+dr, c+dc
        if 0 <= nr < 10 and 0 <= nc < 10 and maze10[nr][nc] != '#':
            yield (nr,nc)

def bfs10(start, goal):
    q = deque([start]); parent = {start: None}; expanded = 0
    while q:
        node = q.popleft(); expanded += 1
        if node == goal:
            p = build_path(parent, goal)
            return p, len(p)-1, expanded
        for nb in maze10_neighbours(node):
            if nb not in parent:
                parent[nb] = node; q.append(nb)
    return None, None, expanded

def ucs10(start, goal):
    pq = [(0, start)]; parent = {start: None}; best = {start: 0}; expanded = 0
    while pq:
        cost, node = heapq.heappop(pq)
        if cost != best[node]: continue
        expanded += 1
        if node == goal:
            p = build_path(parent, goal)
            return p, cost, expanded
        for nb in maze10_neighbours(node):
            step_cost = 5 if maze10[nb[0]][nb[1]] == 'm' else 1
            new_cost = cost + step_cost
            if new_cost < best.get(nb, float('inf')):
                best[nb] = new_cost; parent[nb] = node
                heapq.heappush(pq, (new_cost, nb))
    return None, float('inf'), expanded

p_bfs, moves_bfs, exp_bfs = bfs10(start10, goal10)
p_ucs, cost_ucs, exp_ucs = ucs10(start10, goal10)
print(f"BFS: moves={moves_bfs}, travel cost={sum(5 if maze10[r][c]=='m' else 1 for r,c in p_bfs[1:])}, expanded={exp_bfs}")
print(f"UCS: moves={len(p_ucs)-1}, travel cost={cost_ucs}, expanded={exp_ucs}")
print('BFS path:')
for row in maze10:
    print(' '.join(row))
print('Observation: BFS minimizes moves, while UCS minimizes total cost and can avoid mud even if the route has more moves.')

BFS: moves=18, travel cost=22, expanded=84
UCS: moves=18, travel cost=18, expanded=82
BFS path:
S . . m . . . . . .
. # # m . # # . . .
. . . m . . . m . .
# # . . . # # m . .
. . . m . . . . . .
. m . # # . m . . .
. . . m . . . # # .
# # . . . m . . . .
. . . # # . . m . .
. . . . . . m . . G
Observation: BFS minimizes moves, while UCS minimizes total cost and can avoid mud even if the route has more moves.
